# Diario de Sesiones — Stage 2 corpus run

Normalises every Diario PDF in your Drive corpus into a speech-level table.

Run this in Google Colab rather than locally: the PDFs already live in Drive, and Colab mounts Drive directly, so nothing has to be downloaded. The full corpus is several gigabytes — `períodos 124–143`, with single sessions up to 226 MB — so moving it anywhere else is the slow path.

**Order of cells:** 1 mount → 2 install → 3 pipeline → 4 manifest → 5 batch → 6 QA review.

The batch is resumable. If Colab disconnects, re-run cell 5 and it picks up where it stopped (it skips any file that already has a QA record).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE   = '/content/drive/MyDrive/6. Investigación/Investigación 2026/Vocabulario legislativo'
CORPUS = f'{BASE}/Diarios de sesiones/HCDN_Diarios_124-143'
OUTDIR = f'{BASE}/Stage2_salida'   # created for you; results land here

import os
print('corpus exists:', os.path.isdir(CORPUS))
if os.path.isdir(CORPUS):
    print(sorted(os.listdir(CORPUS))[:25])

If `corpus exists` is False, find the right path. The folder you shared is *owned* by you, so it is under `MyDrive`; print the top level to locate it:

In [ ]:
print(sorted(os.listdir('/content/drive/MyDrive'))[:60])

## 2. Install dependencies

In [ ]:
!pip install -q pymupdf pandas pyarrow

## 3. Load the pipeline

Upload `diario_pipeline.py` to Drive (next to this notebook) and point at it, or use the file picker.

In [ ]:
import shutil, sys, os

SRC = f'{BASE}/diario_pipeline.py'   # where you put the script
if os.path.exists(SRC):
    shutil.copy(SRC, '/content/diario_pipeline.py')
else:
    from google.colab import files
    files.upload()        # choose diario_pipeline.py

sys.path.insert(0, '/content')
import diario_pipeline as P
print('pipeline loaded')

## 4. Build the manifest

One row per PDF. Check the count and the `periodo` spread before committing to the full run — a `periodo` that comes out blank means the folder is not named `Periodo_NNN`, and that column is what the party join keys on later.

In [ ]:
import pandas as pd
from pathlib import Path

os.makedirs(OUTDIR, exist_ok=True)
MANIFEST = Path(OUTDIR) / 'manifest.csv'
n = P.build_manifest(Path(CORPUS), MANIFEST)

man = pd.read_csv(MANIFEST)
print(f'{n} PDFs')
print(man.groupby('periodo', dropna=False).agg(
    files=('path', 'size'), gb=('size_bytes', lambda s: round(s.sum() / 1e9, 2))))

### Smoke test on three files first

Pick one file from the earliest período, one from the middle and one from the latest. If the oratory-size calibration holds and the speech counts look plausible, the full run is safe.

In [ ]:
sample = (man.sort_values('path')
             .groupby('periodo', dropna=False)
             .head(1)
             .sample(min(3, man.periodo.nunique()), random_state=0))

for p in sample.path:
    rows, qa = P.parse_pdf(Path(p))
    print(f"{Path(p).name}: {qa['n_speeches']} speeches, {qa['n_words_total']} words, "
          f"oratory={qa.get('oratory_size')}, purity={qa.get('oratory_size_purity')}, "
          f"status={qa['status']}")

## 5. Run the corpus (resumable)

Writes, per session: `<stem>.parquet`, `<stem>.csv`, and `qa/<stem>.json`. A file that raises is recorded as `status: error` and the batch continues, so one bad PDF never costs you the run.

Re-run this cell after any disconnect.

In [ ]:
P.run_batch(MANIFEST, Path(OUTDIR), resume=True)

## 6. Review the QA summary

Read this before trusting the corpus. What to look for:

- **`status != 'ok'`** — no speaker header was found at all, so nothing was extracted. Inspect those files by hand.
- **`oratory_size` that differs from the corpus mode** — a plate set in a different size. Not wrong in itself, but worth one visual check.
- **`oratory_size_purity` below ~0.95** — speaker headers appearing at two different sizes, which usually means oratory and an insert share a size in that file, so inserts may have leaked into the text.
- **`n_speeches == 0` on a long file** — a layout the heuristics did not recognise.
- **a sharp outlier in words per speech** — a segmentation failure, e.g. several interventions merged.

In [ ]:
qa = pd.read_csv(Path(OUTDIR) / '_qa_summary.csv')
print(qa.status.value_counts(), '\n')
print('oratory sizes:', qa.oratory_size.value_counts(dropna=False).to_dict(), '\n')
print('files needing a look:')
flag = qa[(qa.status != 'ok') | (qa.oratory_size_purity < 0.95) | (qa.n_speeches == 0)]
print(flag[['source_file', 'n_pages', 'n_speeches', 'oratory_size',
            'oratory_size_purity', 'status']].to_string(index=False))

### Assemble the single corpus table

In [ ]:
import glob

parts = [pd.read_parquet(f) for f in sorted(glob.glob(f'{OUTDIR}/*.parquet'))
         if not f.endswith('corpus.parquet')]
corpus = pd.concat(parts, ignore_index=True)
corpus.to_parquet(f'{OUTDIR}/corpus.parquet', index=False)

print(f'{len(corpus):,} speeches  |  {corpus.n_words.sum():,} words  '
      f'|  {corpus.source_file.nunique()} sessions  '
      f'|  {corpus.speaker_key.nunique()} distinct speakers')
print()
print(corpus.groupby('periodo').agg(
    sessions=('source_file', 'nunique'),
    speeches=('speech_seq', 'size'),
    words=('n_words', 'sum')))

### Speaker roster for the party join

Stage 2 leaves `party` blank on purpose. Export the distinct speakers so the bloc roster can be filled in once and joined on `speaker_key` + `periodo`, rather than being guessed per session.

In [ ]:
roster = (corpus[corpus.role.isin(['diputado', 'diputada'])]
          .groupby(['periodo', 'speaker_key', 'speaker_surname', 'speaker_paren'])
          .agg(speeches=('speech_seq', 'size'), words=('n_words', 'sum'))
          .reset_index()
          .sort_values(['periodo', 'speaker_key']))
roster['party'] = ''
roster.to_csv(f'{OUTDIR}/speaker_roster_to_fill.csv', index=False)
print(f'{len(roster)} (período, speaker) pairs -> speaker_roster_to_fill.csv')
roster.head(15)